In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

# List of file paths for the 12 Sentinel-2 bands
file_paths = [
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B01_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B02_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B03_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B04_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B05_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B06_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B07_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B08_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B8A_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B09_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B11_(Raw).tiff",
    "/home/stagiaire/Téléchargements/BR3/2020-08-11-00:00_2020-08-11-23:59_Sentinel-2_L2A_B12_(Raw).tiff"
]


def get_resolution(file_path):
    """
    Get the spatial resolution (pixel size) of a TIFF file.

    Parameters:
        file_path (str): Path to the raster file.

    Returns:
        tuple: (resolution_x, resolution_y) in meters.
    """
    with rasterio.open(file_path) as src:
        resolution_x, resolution_y = src.res
    return resolution_x, resolution_y


# Print spatial resolution for each file
for path in file_paths:
    res_x, res_y = get_resolution(path)
    print(f"Spatial resolution of {path.split('.')[0].split('/')[-1]}: {res_x}m x {res_y}m")


def project_raster(input_file, output_file, target_crs):
    """
    Reproject a raster to a specified coordinate reference system (CRS).

    Parameters:
        input_file (str): Path to the input raster.
        output_file (str): Path to the output reprojected raster.
        target_crs (CRS): Target coordinate reference system.
    """
    with rasterio.open(input_file) as src:
        # Compute transformation and new dimensions
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )

        # Update metadata for the new file
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        # Reproject and save the new raster
        with rasterio.open(output_file, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=Resampling.nearest
                )


# Project each raster to EPSG:32723 (UTM Zone 27S)
for path in file_paths:
    output_path = f"{path.split('.')[0]}_proj.tiff"
    project_raster(path, output_path, CRS.from_epsg(32723))


# Print spatial resolution for each projected file
for path in file_paths:
    projected_path = f"{path.split('.')[0]}_proj.tiff"
    res_x, res_y = get_resolution(projected_path)
    print(f"Spatial resolution of {projected_path.split('/')[-1].split('.')[0]}: {res_x}m x {res_y}m")


def resample_raster(input_file, output_file, target_res_x, target_res_y):
    """
    Resample a raster to a specified spatial resolution.

    Parameters:
        input_file (str): Path to the input raster.
        output_file (str): Path to the output resampled raster.
        target_res_x (float): Desired pixel width in meters.
        target_res_y (float): Desired pixel height in meters.
    """
    with rasterio.open(input_file) as src:
        # Compute new transformation and dimensions
        transform = rasterio.Affine(
            target_res_x, 0.0, src.bounds.left, 0.0, -target_res_y, src.bounds.top
        )
        width = int((src.bounds.right - src.bounds.left) / target_res_x)
        height = int((src.bounds.top - src.bounds.bottom) / target_res_y)

        # Update metadata for the new file
        kwargs = src.meta.copy()
        kwargs.update({
            'transform': transform,
            'width': width,
            'height': height,
            'dtype': 'float32'
        })

        # Resample and save the new raster
        with rasterio.open(output_file, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=src.crs,
                    resampling=Resampling.bilinear
                )


# Resample each band to a 10-meter resolution
for path in file_paths:
    projected_path = f"{path.split('.')[0]}_proj.tiff"
    output_resampled_path = f"{projected_path.split('.')[0]}_10m.tiff"
    resample_raster(projected_path, output_resampled_path, 10, 10)
